# Notebook 05 — 4-Config Experiments & Evaluation

**Configurations:**
| Config | Model | RAG |
|--------|-------|-----|
| A | Base Qwen2.5-3B-Instruct | No |
| B | Base Qwen2.5-3B-Instruct | Yes (top-3 chunks) |
| C | Fine-tuned (LoRA adapter) | No |
| D | Fine-tuned (LoRA adapter) | Yes (top-3 chunks) |

**Metrics:** BLEU (char-level), ROUGE-L, BERTScore (vi), Recall@5 (B/D only)  

> **Prerequisites:** Notebooks 01–04 must be complete.

In [ ]:
from pathlib import Path
import os

# Tự động tìm đường dẫn
if os.path.exists('/content/'):
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/NLP_Final/Source/')
else:
    BASE = "../"

print(f'Base directory: {BASE}')
print(os.listdir(BASE))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base directory: /content/drive/MyDrive/NLP_Final/Source
['.gitignore', 'requirements.txt', 'scripts', '.git', 'data', 'notebooks', 'models', 'results', '.env']


In [ ]:
%%capture
!pip install -q \
    transformers==4.45.0 peft==0.13.0 accelerate==1.0.1 \
    sacrebleu==2.4.3 rouge-score==0.1.2 bert-score==0.3.13
!pip install -q -U bitsandbytes
!pip install -q "numpy<2" faiss-cpu==1.8.0 sentence-transformers==3.2.1

In [ ]:
import importlib
for pkg in ['transformers', 'peft', 'bitsandbytes', 'sentence_transformers']:
    print(f'{pkg}: {importlib.metadata.version(pkg)}')
print('All packages OK.')

transformers: 4.45.0
peft: 0.13.0
bitsandbytes: 0.49.2
sentence_transformers: 3.2.1
All packages OK.


## 1. Configuration & Paths

In [ ]:
import os, json, torch, pickle
import numpy as np

MODEL_ID   = 'Qwen/Qwen2.5-3B-Instruct'
SFT_PATH   = f'{BASE}/models/sft_checkpoint'
INDEX_DIR  = f'{BASE}/data/vector_store/faiss_index'
RESULTS    = f'{BASE}/results'
os.makedirs(RESULTS, exist_ok=True)

SYSTEM_PROMPT = (
    'Bạn là trợ lý AI của Trường Đại học Tôn Đức Thắng (TDTU). '
    'Bạn trả lời các câu hỏi về quy chế, quy định, chính sách của trường một cách chính xác và đầy đủ. '
    'Trả lời bằng tiếng Việt. Nếu không có đủ thông tin, hãy nói rõ điều đó.'
)

print(f'Model: {MODEL_ID}')
print(f'SFT checkpoint: {SFT_PATH}')

Model: Qwen/Qwen2.5-3B-Instruct
SFT checkpoint: /content/drive/MyDrive/NLP_Final/Source/models/sft_checkpoint


## 2. Load Test Dataset

In [ ]:
test_pairs = []
with open(f'{BASE}/data/qa_filtered/qa_test.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        test_pairs.append(json.loads(line.strip()))

print(f'Test pairs: {len(test_pairs)}')
references = [p['answer'] for p in test_pairs]
questions  = [p['question'] for p in test_pairs]

Test pairs: 284


## 3. Load FAISS Retriever

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

faiss_index = faiss.read_index(f'{INDEX_DIR}/index.faiss')
with open(f'{INDEX_DIR}/index.pkl', 'rb') as f:
    index_chunks = pickle.load(f)

embedder = SentenceTransformer('bkai-foundation-models/vietnamese-bi-encoder')

def retrieve(query: str, k: int = 5) -> list[dict]:
    q_vec = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, indices = faiss_index.search(q_vec, k)
    return [{'chunk': index_chunks[i], 'score': float(scores[0][j])}
            for j, i in enumerate(indices[0]) if i < len(index_chunks)]

print(f'FAISS index loaded: {faiss_index.ntotal} vectors')

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

bpe.codes: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

FAISS index loaded: 850 vectors


In [ ]:
# KIỂM TRA NHANH TRƯỚC KHI LOAD
print("=" * 50)
print("CHECKING SETUP...")

# 1. Kiểm tra file adapter tồn tại
import os
if os.path.exists(f'{BASE}/models/sft_checkpoint/adapter_config.json'):
    print("✅ Adapter found")
else:
    print("❌ Adapter NOT found - run notebook 04 first")

CHECKING SETUP...
✅ Adapter found


## 4. Load Models

In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.get_device_properties(0).total_memory > 30e9 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

# Set eos_token cho Qwen2.5
if tokenizer.eos_token != '<|im_end|>':
    tokenizer.eos_token = '<|im_end|>'
    tokenizer.eos_token_id = tokenizer.convert_tokens_to_ids('<|im_end|>')

print(f'<|im_end|> token id: {tokenizer.eos_token_id}')

print('Loading base model...')
_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

print('Wrapping with LoRA adapter...')
ft_model = PeftModel.from_pretrained(_base, SFT_PATH)
ft_model.eval()

# FIX: Đúng cách enable/disable adapter
def use_base():
    """Switch to base model (disable LoRA)"""
    ft_model.disable_adapter_layers()
    # Force model to use base weights
    torch.cuda.empty_cache()

def use_finetuned():
    """Switch to fine-tuned model (enable LoRA)"""
    ft_model.enable_adapter_layers()
    torch.cuda.empty_cache()

print('Model ready.')

Loading tokenizer...
<|im_end|> token id: 151645
Loading base model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Wrapping with LoRA adapter...
Model ready.


In [13]:
# Tối ưu inference speed
print("\n=== OPTIMIZING INFERENCE ===")

# 1. Bật cache (quan trọng nhất!)
ft_model.config.use_cache = True
print("✅ use_cache enabled")

# 2. Giảm max_new_tokens trong generate
# Sửa trong hàm generate: max_new_tokens=128 (thay vì 256)

# 3. Kiểm tra device
print(f"✅ Model on device: {ft_model.device}")

# 4. Clear cache trước khi inference
torch.cuda.empty_cache()
print("✅ Cache cleared")


=== OPTIMIZING INFERENCE ===
✅ use_cache enabled
✅ Model on device: cuda:0
✅ Cache cleared


## 5. Inference Function

In [14]:
def build_prompt(question: str, context_chunks: list[str] = None) -> str:
    if context_chunks:
        context_str = '\n\n'.join([f'[{i+1}] {c}' for i, c in enumerate(context_chunks)])
        user_content = (
            f'Dựa vào các đoạn văn bản sau từ quy chế TDTU:\n\n'
            f'{context_str}\n\nCâu hỏi: {question}'
        )
    else:
        user_content = question
    return (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{user_content}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )


def generate(model, tokenizer, prompt: str, max_new_tokens: int = 256) -> str:  # 256
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return text


print('Inference function ready.')
# Quick smoke test
print("=" * 50)
_prompt = build_prompt('Điều kiện học bổng TDTU là gì?')
print(f'Prompt preview ({len(_prompt)} chars):')
print(_prompt[:200], '...')

Inference function ready.
Prompt preview (337 chars):
<|im_start|>system
Bạn là trợ lý AI của Trường Đại học Tôn Đức Thắng (TDTU). Bạn trả lời các câu hỏi về quy chế, quy định, chính sách của trường một cách chính xác và đầy đủ. Trả lời bằng tiếng Việt.  ...


## 6. Run All 4 Configurations

In [ ]:
from tqdm import tqdm

CONFIGS = {
    'A': {'use_rag': False, 'use_adapter': False, 'label': 'Base LLM, no RAG'},
    'B': {'use_rag': True,  'use_adapter': False, 'label': 'Base LLM + RAG'},
    'C': {'use_rag': False, 'use_adapter': True,  'label': 'Fine-tuned, no RAG'},
    'D': {'use_rag': True,  'use_adapter': True,  'label': 'Fine-tuned + RAG'},
}

all_predictions = {}
all_retrieved   = {}

BATCH_SIZE = 32  # ← Batch size 16

for cfg_name, cfg in CONFIGS.items():
    out_path = f'{RESULTS}/config_{cfg_name}_results.jsonl'

    # Resume: load existing results
    if os.path.exists(out_path):
        preds, ret_ids_per_q = [], []
        with open(out_path, 'r', encoding='utf-8') as f:
            for line in f:
                row = json.loads(line)
                preds.append(row['prediction'])
                ret_ids_per_q.append(row.get('retrieved_chunk_ids', []))
        all_predictions[cfg_name] = preds
        all_retrieved[cfg_name] = ret_ids_per_q
        print(f'Config {cfg_name}: loaded {len(preds)} cached predictions')
        continue

    print(f'\n=== Config {cfg_name}: {cfg["label"]} ===')

    if cfg['use_adapter']:
        use_finetuned()
    else:
        use_base()

    preds, ret_ids_per_q = [], []

    # Batch processing
    num_batches = (len(test_pairs) + BATCH_SIZE - 1) // BATCH_SIZE

    for i in tqdm(range(0, len(test_pairs), BATCH_SIZE), desc=f'Config {cfg_name} (batch={BATCH_SIZE})'):
        batch = test_pairs[i:i+BATCH_SIZE]

        # 1. Retrieval cho cả batch (nếu dùng RAG)
        batch_prompts = []
        batch_top5_ids = []

        for pair in batch:
            if cfg['use_rag']:
                results = retrieve(pair['question'], k=5)
                top3_texts = [r['chunk']['text'] for r in results[:3]]
                top5_ids = [r['chunk']['chunk_id'] for r in results]
                batch_prompts.append(build_prompt(pair['question'], top3_texts))
                batch_top5_ids.append(top5_ids)
            else:
                batch_prompts.append(build_prompt(pair['question']))
                batch_top5_ids.append([])

        # 2. Tokenize batch
        inputs = tokenizer(
            batch_prompts,
            return_tensors='pt',
            truncation=True,
            max_length=1024,
            padding=True  # ← Quan trọng: padding để batch
        ).to(ft_model.device)

        # 3. Generate batch
        with torch.no_grad():
            outputs = ft_model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.05,
            )

        # 4. Decode từng output trong batch
        for j, input_ids in enumerate(inputs['input_ids']):
            generated = outputs[j][len(input_ids):]
            text = tokenizer.decode(generated, skip_special_tokens=True).strip()
            preds.append(text)
            ret_ids_per_q.append(batch_top5_ids[j])

    # Lưu kết quả
    all_predictions[cfg_name] = preds
    all_retrieved[cfg_name] = ret_ids_per_q

    with open(out_path, 'w', encoding='utf-8') as f:
        for pair, pred, ret_ids in zip(test_pairs, preds, ret_ids_per_q):
            f.write(json.dumps({
                'id': pair.get('id'),
                'question': pair['question'],
                'reference': pair['answer'],
                'prediction': pred,
                'retrieved_chunk_ids': ret_ids,
            }, ensure_ascii=False) + '\n')

    print(f'  Saved {len(preds)} predictions → {out_path}')

use_finetuned()
print('\nComplete!')

Config A: loaded 284 cached predictions
Config B: loaded 284 cached predictions
Config C: loaded 284 cached predictions

=== Config D: Fine-tuned + RAG ===


Config D (batch=32):   0%|          | 0/9 [00:00<?, ?it/s]

## 7. Compute Metrics

In [ ]:
from sacrebleu.metrics import BLEU
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

bleu_metric = BLEU(tokenize='char')
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)


def compute_bleu(predictions, references):
    return bleu_metric.corpus_score(predictions, [references]).score


def compute_rougeL(predictions, references):
    scores = [rouge.score(ref, pred)['rougeL'].fmeasure
              for pred, ref in zip(predictions, references)]
    return sum(scores) / len(scores)


def compute_bertscore(predictions, references):
    P, R, F1 = bert_score_fn(
        predictions, references,
        lang='vi',
        model_type='bert-base-multilingual-cased',
        batch_size=8,
        device=device,
        verbose=False,
    )
    return {'P': P.mean().item(), 'R': R.mean().item(), 'F1': F1.mean().item()}


def compute_recall_at_k(pairs, retrieved, k=5):
    annotated = [(pair, ret) for pair, ret in zip(pairs, retrieved) if pair.get('chunk_id')]
    if not annotated:
        print('  WARNING: no chunk_id in test data — Recall@5 skipped')
        return None
    hits = sum(1 for pair, ret_ids in annotated if pair['chunk_id'] in ret_ids[:k])
    return hits / len(annotated)


print('Metric functions ready.')

In [ ]:
summary = {}

for cfg_name, cfg in CONFIGS.items():
    print(f'Computing metrics for Config {cfg_name}...')
    preds  = all_predictions[cfg_name]
    ret_ids = all_retrieved[cfg_name]

    bleu    = compute_bleu(preds, references)
    rougeL  = compute_rougeL(preds, references)
    bscore  = compute_bertscore(preds, references)
    recall5 = compute_recall_at_k(test_pairs, ret_ids) if cfg['use_rag'] else None

    summary[cfg_name] = {
        'label':     cfg['label'],
        'bleu':      round(bleu, 2),
        'rougeL':    round(rougeL * 100, 2),
        'bertscore_P':  round(bscore['P'] * 100, 2),
        'bertscore_R':  round(bscore['R'] * 100, 2),
        'bertscore_F1': round(bscore['F1'] * 100, 2),
        'recall_at_5':  round(recall5 * 100, 2) if recall5 is not None else 'N/A',
    }
    print(f'  BLEU={bleu:.2f}  ROUGE-L={rougeL*100:.2f}  BERTScore-F1={bscore["F1"]*100:.2f}  Recall@5={recall5}')

print('\nAll metrics computed.')

## 8. Results Table

In [ ]:
print('\n' + '='*90)
print(f'{"Config":<8} {"Description":<30} {"BLEU":>7} {"ROUGE-L":>8} {"BERT-F1":>8} {"Recall@5":>10}')
print('='*90)
for cfg_name in ['A', 'B', 'C', 'D']:
    m = summary[cfg_name]
    print(f'{cfg_name:<8} {m["label"]:<30} {m["bleu"]:>7.2f} {m["rougeL"]:>8.2f} '
          f'{m["bertscore_F1"]:>8.2f} {str(m["recall_at_5"]):>10}')
print('='*90)

## 9. Save Evaluation Summary

In [ ]:
SUMMARY_PATH = f'{RESULTS}/eval_summary.json'
with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f'Saved evaluation summary → {SUMMARY_PATH}')

## 10. Human Evaluation Template

In [ ]:
import csv, random

# Sample 50 questions for human evaluation (12-13 per config)
human_eval_samples = []
indices = random.sample(range(len(test_pairs)), min(100, len(test_pairs)))

for idx in indices:
    pair = test_pairs[idx]
    for cfg_name in ['A', 'B', 'C', 'D']:
        human_eval_samples.append({
            'question_id': pair.get('id', f'q{idx}'),
            'question': pair['question'],
            'config': cfg_name,
            'prediction': all_predictions[cfg_name][idx],
            'reference': pair['answer'],
            # Annotator fills these in:
            'accuracy_1_5': '',
            'completeness_1_5': '',
            'fluency_1_5': '',
            'conciseness_1_5': '',
            'no_hallucination_1_5': '',
        })

HUMAN_EVAL_PATH = f'{RESULTS}/human_eval_template.csv'
with open(HUMAN_EVAL_PATH, 'w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=human_eval_samples[0].keys())
    writer.writeheader()
    writer.writerows(human_eval_samples)

print(f'Human evaluation template saved → {HUMAN_EVAL_PATH}')
print(f'Total rows to evaluate: {len(human_eval_samples)}')
print('\nInstructions:')
print('  Rate each response 1-5 on: Accuracy, Completeness, Fluency, Conciseness, No-Hallucination')
print('  (5 = best, 1 = worst)')

## 11. Qualitative Sample Comparison

In [ ]:
# Pick 3 questions and show answers from all 4 configs side-by-side
sample_indices = random.sample(range(len(test_pairs)), 3)

for idx in sample_indices:
    pair = test_pairs[idx]
    print('='*80)
    print(f'Q: {pair["question"]}')
    print(f'Reference: {pair["answer"][:200]}...' if len(pair['answer']) > 200 else f'Reference: {pair["answer"]}')
    print()
    for cfg_name in ['A', 'B', 'C', 'D']:
        pred = all_predictions[cfg_name][idx]
        print(f'[{cfg_name}] {pred[:200]}...' if len(pred) > 200 else f'[{cfg_name}] {pred}')
    print()

---
**Next steps:**
- Fill in `human_eval_template.csv` with scores and import back for analysis
- (Optional) Run `06_rlhf_optional.ipynb` for RLHF extension